In [ ]:

import glob, json, pathlib, sys, traceback
import numpy as np, torch
OUT = {}
try:
    pkg = glob.glob("/kaggle/input/**/casmi/__init__.py", recursive=True)[0]
    sys.path.insert(0, str(pathlib.Path(pkg).parent.parent))
    from casmi import fpnet
    ck = sorted(glob.glob("/kaggle/input/**/fpnet_*.pt", recursive=True))[0]
    st = torch.load(ck, map_location="cpu", weights_only=False)
    OUT["ckpt_type"] = str(type(st))
    if isinstance(st, dict):
        OUT["ckpt_keys"] = list(st)[:12]
        sd = st.get("state_dict") or st.get("model") or st.get("sd") or st
        if isinstance(sd, dict):
            ks = list(sd)
            OUT["n_tensors"] = len(ks)
            OUT["tensor_keys_head"] = ks[:10]
            OUT["bit_like"] = [k for k in ks if any(t in k.lower() for t in ("bit","sel","mask","idx","index"))][:10]
            OUT["buffers_shapes"] = {k: list(v.shape) for k, v in list(sd.items())[:6] if hasattr(v, "shape")}
    m = fpnet.load_model(ck, "cpu")
    OUT["model_dict_keys"] = [k for k in vars(m) if not k.startswith("__")][:20]
    OUT["has_bits"] = hasattr(m, "bits")
except Exception:
    OUT["error"] = traceback.format_exc()[-900:]
pathlib.Path("/kaggle/working/probe.json").write_text(json.dumps(OUT, indent=1, default=str))
print(json.dumps(OUT, indent=1, default=str)[:1500])
